In [1]:
import json
import hashlib
import sys
from urllib.request import build_opener, ProxyHandler

MODELO = "granite4.2:8b-q4_K_M"
API_LOCAL = "http://127.0.0.1:11434"

# Leer los archivos compartidos de 05_Pruebas
with open("../casos_comparacion_v1.json", "rb") as f:
    contenido_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    contenido_config = f.read()

casos = json.loads(contenido_casos)
configuracion = json.loads(contenido_config)

# Verificar que los casos no hayan cambiado
sha_casos = hashlib.sha256(contenido_casos).hexdigest()

assert sha_casos == configuracion["cases_sha256"]
assert [c["case_id"] for c in casos] == [
    "S-005", "S-006", "S-007",
    "S-008", "S-009", "S-010"
]

assert configuracion["api_url"] == API_LOCAL + "/api/chat"

# Conectarse directamente al Ollama local
cliente_local = build_opener(ProxyHandler({}))

with cliente_local.open(API_LOCAL + "/api/version", timeout=10) as r:
    version_ollama = json.load(r)["version"]

print("Python:", sys.version.split()[0])
print("Ollama:", version_ollama)
print("Modelo seleccionado:", MODELO)
print("Casos verificados:", len(casos))
print("SHA256:", sha_casos)
print("Prompt:", configuracion["prompt_version"])
print("Archivos compartidos: correctos")

Python: 3.11.16
Ollama: 0.40.13
Modelo seleccionado: granite4.2:8b-q4_K_M
Casos verificados: 6
SHA256: f73d159bb43cc9c93d2768e997052f4381a2c24bb046c82826b1793c1b8da37d
Prompt: V3
Archivos compartidos: correctos


In [2]:
import copy
import json
import time
from urllib.request import Request

# Caso diagnóstico: no pertenece a S-005 a S-010
contenido = (
    "Autores: Autor X y Autor Y.\n"
    "Predicción recibida: favorable (simulada).\n"
    "Modelo predictor y puntuación: no disponibles.\n"
    "[E1] Autor X tiene un artículo sobre bases de datos académicas.\n"
    "[E2] Autor Y tiene un artículo sobre recuperación de información.\n"
    "No se proporcionan datos de colaboración previa."
)

# Reutilizar V3 y adaptar únicamente los parámetros de Granite
solicitud_granite = copy.deepcopy(
    configuracion["request_defaults"]
)

solicitud_granite["model"] = MODELO
solicitud_granite["think"] = False

solicitud_granite["options"].update({
    "num_ctx": 4096,
    "num_predict": 2048,
    "temperature": 1.0,
    "top_p": 0.95
})

solicitud_granite["messages"] = [
    {
        "role": "system",
        "content": configuracion["system_prompt"]
    },
    {
        "role": "user",
        "content": configuracion["user_prompt_template"].format(
            case_id="DIAG-G01",
            contenido=contenido
        )
    }
]

# Enviar solicitud a la API local
peticion = Request(
    configuracion["api_url"],
    data=json.dumps(
        solicitud_granite, ensure_ascii=False
    ).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST"
)

inicio = time.perf_counter()

with cliente_local.open(peticion, timeout=180) as conexion:
    respuesta_granite = json.load(conexion)

tiempo_cliente = time.perf_counter() - inicio

# Mostrar respuesta original y métricas
print("MODELO:", respuesta_granite["model"])
print("\nRESPUESTA:\n")
print(respuesta_granite["message"]["content"])

print("\nMÉTRICAS:")
print("Tiempo cliente:", round(tiempo_cliente, 3), "s")

for campo in (
    "total_duration",
    "load_duration",
    "prompt_eval_duration",
    "eval_duration"
):
    valor = respuesta_granite.get(campo)
    if valor is not None:
        print(campo, round(valor / 1e9, 3), "s")

for campo in (
    "prompt_eval_count",
    "prompt_eval_cached_count",
    "eval_count",
    "done_reason"
):
    print(campo, respuesta_granite.get(campo))

print(
    "Razonamiento devuelto:",
    bool(respuesta_granite["message"].get("thinking"))
)


MODELO: granite4.2:8b-q4_K_M

RESPUESTA:

**Evidencia**  
[E1] Autor X tiene un artículo sobre bases de datos académicas.  
[E2] Autor Y tiene un artículo sobre recuperación de información.  

No se proporciona ninguna evidencia de colaboración previa entre Autor X y Autor Y (por ejemplo, publicaciones conjuntas, coautorías en el mismo trabajo, proyecto compartido, etc.).  

**Interpretación**  
La evidencia disponible muestra únicamente que Autor X ha publicado sobre bases de datos académicas y que Autor Y ha publicado sobre recuperación de información. Ambos temas están relacionados con la gestión y búsqueda de información en contextos académicos, lo que puede generar una similitud temática superficial. Sin embargo, la similitud temática por sí sola no constituye, según las reglas proporcionadas, evidencia de que los autores hayan trabajado juntos en el pasado, compartan un proyecto actual o colaboren en el futuro. La predicción recibida es “favorable” (simulada), pero la evidencia n

In [3]:
print("Tokens de salida:", respuesta_granite.get("eval_count"))
print("Terminación:", respuesta_granite.get("done_reason"))
print("Razonamiento solicitado:", solicitud_granite.get("think"))
print("Razonamiento devuelto:", bool(
    respuesta_granite.get("message", {}).get("thinking")
))


Tokens de salida: 661
Terminación: stop
Razonamiento solicitado: False
Razonamiento devuelto: False


In [4]:
import json
import copy
import hashlib
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request

# 1. Verificar los archivos compartidos
with open("../casos_comparacion_v1.json", "rb") as f:
    datos_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    datos_config = f.read()

casos = json.loads(datos_casos)
config = json.loads(datos_config)

sha_casos = hashlib.sha256(datos_casos).hexdigest()
sha_config = hashlib.sha256(datos_config).hexdigest()

assert sha_casos == config["cases_sha256"]
assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)

assert [c["case_id"] for c in casos] == [
    "S-005", "S-006", "S-007",
    "S-008", "S-009", "S-010"
]

assert MODELO == "granite4.2:8b-q4_K_M"
assert respuesta_granite.get("done_reason") == "stop"
assert not respuesta_granite.get("message", {}).get("thinking")

# 2. Verificar el modelo instalado
with cliente_local.open(API_LOCAL + "/api/tags", timeout=10) as f:
    modelos = json.load(f)["models"]

modelo_info = next(
    m for m in modelos if m["name"] == MODELO
)

with cliente_local.open(API_LOCAL + "/api/version", timeout=10) as f:
    version_ollama = json.load(f)["version"]

# 3. Revisar registros existentes para no duplicarlos
try:
    with open("resultados.jsonl", encoding="utf-8") as f:
        existentes = [json.loads(line) for line in f if line.strip()]
except FileNotFoundError:
    existentes = []

for r in existentes:
    if r.get("solicitud", {}).get("model") != MODELO:
        raise ValueError("El archivo contiene resultados de otro modelo.")

registrados = {r["case_id"] for r in existentes}

# 4. Guardar todas las ejecuciones en un único archivo
with open("resultados.jsonl", "a", encoding="utf-8") as salida:

    # Conservar el diagnóstico que ya realizamos
    if "DIAG-G01" not in registrados:
        diagnostico = {
            "execution_id": str(uuid4()),
            "case_id": "DIAG-G01",
            "grupo": "diagnostico",
            "fecha_registro_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "modelo_digest": modelo_info["digest"],
            "ollama_version": version_ollama,
            "solicitud": solicitud_granite,
            "respuesta_original": respuesta_granite,
            "tiempo_cliente_s": round(tiempo_cliente, 3)
        }

        salida.write(json.dumps(
            diagnostico, ensure_ascii=False
        ) + "\n")
        salida.flush()
        print("DIAG-G01 | diagnóstico guardado")

    # Ejecutar los seis casos
    for caso in casos:
        case_id = caso["case_id"]

        if case_id in registrados:
            print(case_id, "| ya registrado")
            continue

        solicitud = copy.deepcopy(config["request_defaults"])

        solicitud["model"] = MODELO
        solicitud["think"] = False

        # Configuración específica de Granite 4.2
        solicitud["options"].update({
            "num_ctx": 4096,
            "num_predict": 2048,
            "temperature": 1.0,
            "top_p": 0.95
        })

        solicitud["messages"] = [
            {
                "role": "system",
                "content": config["system_prompt"]
            },
            {
                "role": "user",
                "content": config["user_prompt_template"].format(
                    case_id=case_id,
                    contenido=caso["contenido"]
                )
            }
        ]

        registro = {
            "execution_id": str(uuid4()),
            "case_id": case_id,
            "grupo": "comparacion",
            "configuracion_modelo": "granite_directa_v1",
            "prompt_version": "V3",
            "modelo_digest": modelo_info["digest"],
            "ollama_version": version_ollama,
            "sha256_casos": sha_casos,
            "sha256_configuracion": sha_config,
            "fecha_inicio_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "solicitud": solicitud,
            "reintentos": 0
        }

        inicio = time.perf_counter()

        try:
            peticion = Request(
                config["api_url"],
                data=json.dumps(
                    solicitud, ensure_ascii=False
                ).encode("utf-8"),
                headers={"Content-Type": "application/json"},
                method="POST"
            )

            with cliente_local.open(
                peticion, timeout=config["timeout_s"]
            ) as conexion:
                respuesta = json.load(conexion)

            registro["respuesta_original"] = respuesta

            if respuesta.get("done_reason") == "length":
                registro["estado"] = "truncada"
            elif respuesta.get("message", {}).get("thinking"):
                registro["estado"] = "revisar_modo"
            elif (
                respuesta.get("done_reason") == "stop"
                and respuesta.get("message", {}).get("content")
            ):
                registro["estado"] = "ok"
            else:
                registro["estado"] = "revisar"

        except Exception as error:
            registro["estado"] = "error"
            registro["error_tipo"] = type(error).__name__
            registro["error"] = str(error)

        registro["tiempo_cliente_s"] = round(
            time.perf_counter() - inicio, 3
        )
        registro["fecha_fin_utc"] = datetime.now(
            timezone.utc
        ).isoformat()

        # Guardar incluso cuando haya errores
        salida.write(json.dumps(
            registro, ensure_ascii=False
        ) + "\n")
        salida.flush()

        respuesta = registro.get("respuesta_original", {})

        print(
            case_id,
            "|", registro["estado"],
            "|", respuesta.get("eval_count", "-"), "tokens",
            "|", registro["tiempo_cliente_s"], "s"
        )

print("\nArchivo actualizado: resultados.jsonl")


DIAG-G01 | diagnóstico guardado
S-005 | ok | 630 tokens | 10.883 s
S-006 | ok | 689 tokens | 11.859 s
S-007 | ok | 808 tokens | 14.578 s
S-008 | ok | 740 tokens | 12.021 s
S-009 | ok | 755 tokens | 11.787 s
S-010 | ok | 681 tokens | 10.359 s

Archivo actualizado: resultados.jsonl


In [5]:
import json
import copy
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request, build_opener, ProxyHandler

# Recuperar el diagnóstico original
with open("resultados.jsonl", encoding="utf-8") as f:
    registros = [json.loads(line) for line in f if line.strip()]

assert not any(r["case_id"] == "DIAG-G02" for r in registros), (
    "DIAG-G02 ya existe. Revisar antes de repetir."
)

anterior = next(r for r in registros if r["case_id"] == "DIAG-G01")

# Misma evidencia y prompt; cambiar solo el modo y presupuesto
solicitud = copy.deepcopy(anterior["solicitud"])
solicitud["think"] = True
solicitud["options"]["num_ctx"] = 8192
solicitud["options"]["num_predict"] = 4096

registro = {
    "execution_id": str(uuid4()),
    "case_id": "DIAG-G02",
    "grupo": "diagnostico",
    "configuracion_modelo": "granite_thinking_v1",
    "modelo_digest": anterior["modelo_digest"],
    "ollama_version": anterior["ollama_version"],
    "fecha_inicio_utc": datetime.now(timezone.utc).isoformat(),
    "solicitud": solicitud
}

cliente = build_opener(ProxyHandler({}))
inicio = time.perf_counter()

try:
    peticion = Request(
        "http://127.0.0.1:11434/api/chat",
        data=json.dumps(solicitud, ensure_ascii=False).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    with cliente.open(peticion, timeout=300) as conexion:
        respuesta = json.load(conexion)

    registro["respuesta_original"] = respuesta

    if respuesta.get("done_reason") == "length":
        registro["estado"] = "truncada"
    elif respuesta.get("done_reason") == "stop" and respuesta.get("message", {}).get("content"):
        registro["estado"] = "ok"
    else:
        registro["estado"] = "revisar"

except Exception as error:
    registro["estado"] = "error"
    registro["error"] = str(error)

registro["tiempo_cliente_s"] = round(time.perf_counter() - inicio, 3)
registro["fecha_fin_utc"] = datetime.now(timezone.utc).isoformat()

# Conservar incluso fallos y truncamientos
with open("resultados.jsonl", "a", encoding="utf-8") as f:
    f.write(json.dumps(registro, ensure_ascii=False) + "\n")

print("Estado:", registro["estado"])
print("Tiempo cliente:", registro["tiempo_cliente_s"], "s")

if "respuesta_original" in registro:
    r = registro["respuesta_original"]
    print("Tokens entrada:", r.get("prompt_eval_count"))
    print("Tokens generados:", r.get("eval_count"))
    print("Terminación:", r.get("done_reason"))
    print("Razonamiento separado:", bool(r["message"].get("thinking")))
    print("\nRESPUESTA FINAL:\n")
    print(r["message"].get("content", ""))
else:
    print("Error:", registro.get("error"))

print("\nRegistro añadido a resultados.jsonl")


Estado: ok
Tiempo cliente: 26.963 s
Tokens entrada: 433
Tokens generados: 1404
Terminación: stop
Razonamiento separado: True

RESPUESTA FINAL:

Evidencia  
[E1] Autor X tiene un artículo sobre bases de datos académicas. [E2] Autor Y tiene un artículo sobre recuperación de información. No se proporcionan datos de colaboración previa (ni publicaciones conjuntas, ni proyectos compartidos, ni interacciones documentadas). La evidencia consta únicamente de estos dos hechos individuales sobre los temas de los artículos de cada autor.  

Interpretación  
La evidencia muestra que Autor X ha publicado sobre bases de datos académicas y que Autor Y ha publicado sobre recuperación de información. Estos temas pueden intersectar teóricamente (por ejemplo, recuperación de información a partir de bases de datos), pero la evidencia no menciona ninguna colaboración previa (ni artículos conjuntos, ni proyecto compartido en curso, ni comunicación directa relacionada con un proyecto común). La similitud tem

In [6]:
import json
import copy
import hashlib
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request, build_opener, ProxyHandler

# 1. Leer archivos compartidos
with open("../casos_comparacion_v1.json", "rb") as f:
    datos_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    datos_config = f.read()

casos = json.loads(datos_casos)
config = json.loads(datos_config)

sha_casos = hashlib.sha256(datos_casos).hexdigest()
sha_config = hashlib.sha256(datos_config).hexdigest()

assert sha_casos == config["cases_sha256"]
assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)

# 2. Leer resultados existentes
with open("resultados.jsonl", encoding="utf-8") as f:
    anteriores = [json.loads(line) for line in f if line.strip()]

directos = {
    r["case_id"]: r for r in anteriores
    if r.get("configuracion_modelo") == "granite_directa_v1"
    and r.get("grupo") == "comparacion"
}

assert len(directos) == 6
assert set(directos) == {c["case_id"] for c in casos}

diagnostico = next(
    r for r in anteriores if r["case_id"] == "DIAG-G02"
)

assert diagnostico["estado"] == "ok"
assert diagnostico["solicitud"]["think"] is True
assert diagnostico["respuesta_original"]["message"].get("thinking")

# Evitar repetir casos de la misma configuración
registrados = {
    r["case_id"] for r in anteriores
    if r.get("configuracion_modelo") == "granite_thinking_v1"
    and r.get("grupo") == "comparacion"
}

cliente = build_opener(ProxyHandler({}))

# 3. Ejecutar los seis casos secuencialmente
with open("resultados.jsonl", "a", encoding="utf-8") as archivo:

    for caso in casos:
        case_id = caso["case_id"]

        if case_id in registrados:
            print(case_id, "| ya registrado")
            continue

        # Copiar exactamente la solicitud del modo directo
        solicitud = copy.deepcopy(directos[case_id]["solicitud"])

        # Verificar evidencia y prompt original
        assert solicitud["messages"][0]["content"] == config["system_prompt"]
        assert solicitud["messages"][1]["content"] == (
            config["user_prompt_template"].format(
                case_id=case_id,
                contenido=caso["contenido"]
            )
        )

        # Misma configuración usada en DIAG-G02
        solicitud["think"] = True
        solicitud["options"]["num_ctx"] = 8192
        solicitud["options"]["num_predict"] = 4096

        assert solicitud["options"]["temperature"] == 1.0
        assert solicitud["options"]["top_p"] == 0.95

        registro = {
            "execution_id": str(uuid4()),
            "case_id": case_id,
            "grupo": "comparacion",
            "configuracion_modelo": "granite_thinking_v1",
            "prompt_version": "V3",
            "modelo_digest": diagnostico["modelo_digest"],
            "ollama_version": diagnostico["ollama_version"],
            "sha256_casos": sha_casos,
            "sha256_configuracion": sha_config,
            "fecha_inicio_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "solicitud": solicitud,
            "reintentos": 0
        }

        inicio = time.perf_counter()

        try:
            peticion = Request(
                config["api_url"],
                data=json.dumps(
                    solicitud, ensure_ascii=False
                ).encode("utf-8"),
                headers={"Content-Type": "application/json"},
                method="POST"
            )

            with cliente.open(peticion, timeout=300) as respuesta_api:
                respuesta = json.load(respuesta_api)

            registro["respuesta_original"] = respuesta

            if respuesta.get("done_reason") == "length":
                registro["estado"] = "truncada"
            elif (
                respuesta.get("done_reason") == "stop"
                and respuesta.get("message", {}).get("content")
                and respuesta.get("message", {}).get("thinking")
            ):
                registro["estado"] = "ok"
            else:
                registro["estado"] = "revisar"

        except Exception as error:
            registro["estado"] = "error"
            registro["error_tipo"] = type(error).__name__
            registro["error"] = str(error)

        registro["tiempo_cliente_s"] = round(
            time.perf_counter() - inicio, 3
        )
        registro["fecha_fin_utc"] = datetime.now(
            timezone.utc
        ).isoformat()

        # Guardar la respuesta completa, incluido el razonamiento
        archivo.write(
            json.dumps(registro, ensure_ascii=False) + "\n"
        )
        archivo.flush()

        r = registro.get("respuesta_original", {})

        print(
            case_id,
            "|", registro["estado"],
            "|", r.get("eval_count", "-"), "tokens",
            "|", registro["tiempo_cliente_s"], "s"
        )

print("\nArchivo actualizado: resultados.jsonl")


S-005 | ok | 1390 tokens | 28.182 s
S-006 | ok | 2101 tokens | 39.295 s
S-007 | ok | 3528 tokens | 70.927 s
S-008 | ok | 3109 tokens | 62.203 s
S-009 | ok | 2727 tokens | 55.74 s
S-010 | ok | 3665 tokens | 73.993 s

Archivo actualizado: resultados.jsonl
